# `parse_word` + `apply_changes` — pattern extract / modify / rebuild

Module testé : [`src/docpipeline/parsing/word/parse_word.py`](../src/docpipeline/parsing/word/parse_word.py).

## Le pattern transversal du projet

Pour traduction, conversion, redaction — partout où on doit modifier du texte SANS perdre les styles :

```
extract  : parse_word(docx)             → spans avec styles + span_id stable
modify   : on change SEULEMENT le text de chaque span (translate, redact, …)
rebuild  : apply_changes(docx_in, {span_id: new_text}, docx_out)
           → reconstruit le .docx avec nouveaux textes mais styles intacts
```

Le `span_id` (format `w_<para_idx>_<run_idx>`) est la **clé stable** qui fait le pont entre extract et rebuild — robuste aux textes dupliqués.

## Démo : cas d'usage **traduction FR → EN**

On parse `tests/fixtures/contrat_assurance.docx` (un contrat d'assurance en français), on traduit chaque span via un mini-dictionnaire FR→EN (sans LLM pour la démo — en prod ce serait un appel LLM, autorisé par `CLAUDE.md` dans la couche translation), et on reconstruit le `.docx` traduit.

**Le fichier traduit est sauvegardé dans `notebooks/_outputs/word_translation/contrat_assurance_EN_js.docx`** pour que vous puissiez l'ouvrir avec Word et inspecter visuellement que les styles sont préservés.

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import json
from pathlib import Path
from docpipeline.parsing.word.parse_word import parse_word, apply_changes

DOCX_FR = Path('../tests/fixtures/contrat_assurance.docx')
DOCX_EN = Path('_outputs/word_translation/contrat_assurance_EN_js.docx')
DOCX_EN.parent.mkdir(parents=True, exist_ok=True)

# 1) EXTRACT
result = parse_word(DOCX_FR)
print('=== doc_summary ===')
print(json.dumps(result['doc_summary'], indent=2, ensure_ascii=False, default=str))
print()
print(f"=== span_df ({len(result['span_df'])} spans) ===")
print(result['span_df'][['span_id', 'text', 'font_name', 'font_size_pt', 'bold', 'italic']].to_string(index=False))

# 2) MODIFY : traduction FR -> EN via dictionnaire (zero LLM pour la demo)
FR_TO_EN = {
    "Contrat d'assurance \u2014 Conditions G\u00e9n\u00e9rales": "Insurance Contract \u2014 General Conditions",
    "1. Objet du contrat": "1. Purpose of the contract",
    "Le pr\u00e9sent contrat (IA) couvre les accidents individuels survenus dans le cadre de l'activit\u00e9 professionnelle de l'assur\u00e9. La garantie BI (Business Interruption) est applicable selon l'article 5.": "This contract (IA) covers individual accidents occurring as part of the insured's professional activity. The BI (Business Interruption) coverage applies under article 5.",
    "2. Garanties": "2. Coverage",
    "2.1 Garantie principale": "2.1 Primary coverage",
    "La franchise applicable est fix\u00e9e \u00e0 300 euros par sinistre. Le plafond d'indemnisation est de 50 000 euros par ann\u00e9e civile.": "The applicable deductible is set at 300 euros per claim. The compensation limit is 50,000 euros per calendar year.",
    "IMPORTANT : ": "IMPORTANT: ",
    "Toute d\u00e9claration de sinistre doit intervenir dans un d\u00e9lai de 5 jours ouvr\u00e9s.": "Any claim must be reported within 5 business days.",
    "3. Tableau des garanties": "3. Coverage table",
    "4. Exclusions": "4. Exclusions",
    "Sont exclus du pr\u00e9sent contrat les sinistres r\u00e9sultant de : faute intentionnelle, guerre, actes terroristes, catastrophes naturelles non d\u00e9clar\u00e9es.": "The following claims are excluded from this contract: intentional fault, war, terrorism, undeclared natural disasters.",
}

changes = {row['span_id']: FR_TO_EN[row['text']]
           for _, row in result['span_df'].iterrows()
           if row['text'] in FR_TO_EN}
print()
print(f'=== Traductions appliquees : {len(changes)} / {len(result["span_df"])} spans ===')
for sid, tr in changes.items():
    print(f'  {sid} -> {tr[:80]}')

# 3) REBUILD
apply_changes(DOCX_FR, changes, DOCX_EN)
print()
print(f'>>> Fichier traduit ecrit : {DOCX_EN}')
print(f'    Taille : {DOCX_EN.stat().st_size} bytes')

# 4) VERIF : on re-parse le .docx traduit et on compare les styles avec l'original
result_en = parse_word(DOCX_EN)
before = result['span_df']
after  = result_en['span_df']

print()
print('=== Comparaison FR / EN sur les 5 premiers spans ===')
for i in range(min(5, len(before))):
    b = before.iloc[i]
    a = after.iloc[i]
    print(f"  {b['span_id']} :")
    print(f"    FR : {b['text']}")
    print(f"    EN : {a['text']}")

print()
print('=== Styles preserves apres traduction ? ===')
for col in ['font_name', 'font_size_pt', 'bold', 'italic', 'underline', 'color', 'highlight']:
    same = (before[col].fillna('').astype(str) == after[col].fillna('').astype(str)).all()
    print(f'  {col:15s} : {"OK" if same else "DIFF"}')

=== doc_summary ===
{
  "doc_hash": "0134552717eb2b56e4ae7b799c060f7e70ed838968e293f3c9ea6c5faa9eb5c2",
  "file_size_bytes": 37367,
  "n_paragraphs": 10,
  "n_spans": 11,
  "n_images": 0,
  "n_tables": 1,
  "n_sections": 1,
  "n_headings": 6,
  "n_list_items": 0,
  "total_char_count": 689,
  "source_tool": "Microsoft Word",
  "metadata": {
    "title": "",
    "author": "Faseya Test",
    "last_modified_by": "",
    "subject": "",
    "keywords": "",
    "category": "",
    "comments": "generated by python-docx",
    "revision": 1,
    "created": "2013-12-23T23:15:00+00:00",
    "modified": "2013-12-23T23:15:00+00:00",
    "last_printed": null,
    "version": "",
    "content_status": "",
    "language": ""
  },
  "has_toc": true,
  "has_track_changes": false,
  "has_comments": false,
  "has_footnotes": false,
  "has_hyperlinks": false,
  "style_counts": {
    "Heading 1": 4,
    "Normal": 4,
    "Title": 1,
    "Heading 2": 1
  },
  "recommended_strategy": "use_native_extraction",
  "